# Customer-Level Master: Cleaning

**Input:**  `customer_level_master_table.csv` (one row per `customer_unique_id`, built by
`data-visualization/data-processing-customer_master.ipynb`).
**Output:** `cleaned_customer_level_master_table.csv` (same grain, analysis-ready), written next to
this notebook in `data_cleaning/` — mirroring `cleaning_master_order_table.ipynb`.

The upstream master is already well-formed, so cleaning here is deliberately *light and documented*.
We do five things and nothing more:

1. Remove a redundant column (`customer_id_count`, an exact duplicate of `customer_orders`).
2. Parse the two date columns to real datetimes.
3. Make structural missingness explicit with boolean coverage flags — we **never impute**
   satisfaction or delivery facts.
4. Flag (not drop) the 3 zero-value payment rows and standardise the city/state text.
5. Add a winsorised spend column purely as a plotting aid (raw spend is preserved untouched).

Every step prints a before/after check so the cleaning is auditable.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the raw master robustly, whether this runs from repo root or from data_cleaning/
here = Path.cwd()
search = [here, here.parent, here / 'data-visualization', here.parent / 'data-visualization']
src = next((p / 'customer_level_master_table.csv' for p in search
           if (p / 'customer_level_master_table.csv').exists()), None)
if src is None:
    raise FileNotFoundError('Could not locate customer_level_master_table.csv')

# Write the cleaned file into this notebook's folder (data_cleaning/) when possible
out_dir = here if here.name == 'data_cleaning' else (here / 'data_cleaning'
          if (here / 'data_cleaning').exists() else here)
out_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(src)
print(f'Source : {src}')
print(f'Output : {out_dir / "cleaned_customer_level_master_table.csv"}')
print(f'Loaded {len(df):,} rows x {df.shape[1]} columns')
before_shape = df.shape

Source : C:\Users\iampr\Downloads\ecommerce-visual-analytics-main\ecommerce-visual-analytics-main\data-visualization\customer_level_master_table.csv
Output : C:\Users\iampr\Downloads\ecommerce-visual-analytics-main\ecommerce-visual-analytics-main\data_cleaning\cleaned_customer_level_master_table.csv
Loaded 96,096 rows x 23 columns


## Step 0 — Integrity checks (must hold before we touch anything)

In [2]:
assert df['customer_unique_id'].is_unique, 'grain broken: duplicate customer_unique_id'
dup_cols_identical = (df['customer_orders'] == df['customer_id_count']).all()
print('customer_unique_id is unique      :', df['customer_unique_id'].is_unique)
print('customer_orders == customer_id_count:', dup_cols_identical)
print('states present                    :', df['customer_state'].nunique(), '(Brazil has 27)')

customer_unique_id is unique      : True
customer_orders == customer_id_count: True
states present                    : 27 (Brazil has 27)


## Step 1 — Drop the redundant column

`customer_id_count` is an exact copy of `customer_orders` (verified above), so it carries no
information.

In [3]:
if dup_cols_identical and 'customer_id_count' in df.columns:
    df = df.drop(columns='customer_id_count')
print('columns now:', df.shape[1])

columns now: 22


## Step 2 — Parse dates

Sanity-check the range against the known dataset window (Sep 2016 - Oct 2018).

In [4]:
for col in ['first_order_date', 'last_order_date']:
    df[col] = pd.to_datetime(df[col], errors='coerce')
print('first_order_date range:', df['first_order_date'].min(), '->', df['first_order_date'].max())
print('unparseable dates     :', df[['first_order_date', 'last_order_date']].isna().sum().to_dict())

first_order_date range: 2016-09-04 21:15:19 -> 2018-10-17 17:30:18
unparseable dates     : {'first_order_date': 0, 'last_order_date': 0}


## Step 3 — Make missingness explicit (coverage flags, no imputation)

| Column | Null count | Meaning |
|---|---|---|
| `avg_review_score` | 716 | customer never left a review |
| `avg_delivery_delay_days`, `avg_actual_delivery_days` | 2,740 | no order was ever delivered |

`has_review` / `has_delivery` let every downstream view filter honestly instead of treating
"no delivery" as "on time".

In [5]:
df['has_review'] = df['review_count'].gt(0)
df['has_delivery'] = df['avg_actual_delivery_days'].notna()
print('has_review  :', df['has_review'].sum(), f'({df["has_review"].mean()*100:.1f}%)')
print('has_delivery:', df['has_delivery'].sum(), f'({df["has_delivery"].mean()*100:.1f}%)')
assert (df['avg_review_score'].isna() == (df['review_count'] == 0)).all()
print('review-null <-> review_count==0 consistent: True')

has_review  : 95380 (99.3%)
has_delivery: 93356 (97.1%)
review-null <-> review_count==0 consistent: True


## Step 4 — Flag zero-payment rows and standardise text

Three customers have `total_payment_value == 0` (voucher-only or an upstream artifact). We flag,
not drop, so they can be excluded from revenue math while remaining in counts.

In [6]:
df['zero_payment_flag'] = df['total_payment_value'].eq(0)
print('zero-payment customers:', int(df['zero_payment_flag'].sum()))
df['customer_city'] = df['customer_city'].str.strip().str.lower()
df['customer_state'] = df['customer_state'].str.strip().str.upper()
print('cities (standardised) :', df['customer_city'].nunique())

zero-payment customers: 3
cities (standardised) : 4119


## Step 5 — Winsorised spend column (visualisation aid only)

Raw `total_payment_value` is preserved; `total_payment_value_capped` is clipped at the 99th
percentile so distribution plots are not dominated by a handful of extreme spenders.

In [7]:
p99 = df['total_payment_value'].quantile(0.99)
df['total_payment_value_capped'] = df['total_payment_value'].clip(upper=p99)
print(f'99th percentile spend: R$ {p99:,.2f}')
print(f'customers above cap  : {(df["total_payment_value"] > p99).sum():,}')

99th percentile spend: R$ 1,122.46
customers above cap  : 961


## Step 6 — Final validation and write

In [8]:
assert len(df) == before_shape[0], 'row count changed during cleaning!'
summary = pd.DataFrame({
    'null_count': df.isna().sum(),
    'null_pct': (df.isna().mean() * 100).round(2),
    'dtype': df.dtypes.astype(str),
})
print('Rows:', len(df), '| Columns:', df.shape[1], f'(was {before_shape[1]})')
summary

Rows: 96096 | Columns: 26 (was 23)


,null_count,null_pct,dtype
customer_unique_id,0,0.00,str
customer_orders,0,0.00,int64
first_order_date,0,0.00,datetime64[us]
last_order_date,0,0.00,datetime64[us]
customer_city,0,0.00,str
customer_state,0,0.00,str
total_item_value,0,0.00,float64
total_freight_value,0,0.00,float64
total_payment_value,0,0.00,float64
avg_payment_value,1,0.00,float64


In [9]:
out = out_dir / 'cleaned_customer_level_master_table.csv'
df.to_csv(out, index=False)
print('wrote', out)
print('new columns added:', ['has_review', 'has_delivery', 'zero_payment_flag',
                             'total_payment_value_capped'])
print('columns dropped  :', ['customer_id_count'])

wrote C:\Users\iampr\Downloads\ecommerce-visual-analytics-main\ecommerce-visual-analytics-main\data_cleaning\cleaned_customer_level_master_table.csv
new columns added: ['has_review', 'has_delivery', 'zero_payment_flag', 'total_payment_value_capped']
columns dropped  : ['customer_id_count']


### Cleaning log (for the report / appendix)

| # | Action | Rows affected | Rationale |
|---|---|---|---|
| 1 | Dropped `customer_id_count` | column | Exact duplicate of `customer_orders` |
| 2 | Parsed `first_order_date`, `last_order_date` | all | Reliable tenure/cohort logic |
| 3 | Added `has_review` flag | 716 without a review | Explicit review coverage, no imputation |
| 4 | Added `has_delivery` flag | 2,740 never delivered | Never treat "no delivery" as on-time |
| 5 | Added `zero_payment_flag` | 3 zero-value | Exclude from revenue math, keep for counts |
| 6 | Standardised city/state text | all | Consistent grouping |
| 7 | Added `total_payment_value_capped` (p99) | ~1% above cap | Robust plots; raw preserved |

Nothing was imputed or deleted; the cleaned table keeps the same 96,096-row grain.